# 02 · Limpieza de los datos

**Entrada:** `data/raw/*.csv` · **Salida:** `data/processed/` (un archivo limpio por región y uno combinado).

Pasos, en este orden (cada uno depende del anterior):

| # | Función (`src/cleaning.py`) | Qué hace |
|---|---|---|
| 1 | `unificar_duplicados` | Una fila por hora (promedia las repetidas) |
| 2 | `marcar_valores_sospechosos` | Convierte en vacío los valores `<= 0` |
| 3 | `reindexar_calendario_horario` | Agrega las horas ausentes como filas vacías |
| 4 | `recortar_extremos_vacios` | Quita los vacíos del inicio y del final (no se pueden estimar) |
| 5 | `interpolar_huecos` | Rellena los vacíos de hasta `LIMITE_HORAS` horas seguidas |
| 6 | `validar_limpieza` | Comprueba que la serie quedó regular y sin vacíos |

## 1. Configuración y carga

In [1]:
%load_ext autoreload
%autoreload 2

import io
from contextlib import redirect_stdout

import pandas as pd

from src.config import PROCESSED_DIR
from src.data_loader import load_regions_dict
from src.cleaning import (
    unificar_duplicados,
    marcar_valores_sospechosos,
    reindexar_calendario_horario,
    recortar_extremos_vacios,
    interpolar_huecos,
    validar_limpieza,
)

LIMITE_HORAS = 2   # máximo de horas seguidas que se interpolan

In [2]:
regiones = load_regions_dict()   # {region: DataFrame}
print(f"{len(regiones)} regiones cargadas")

12 regiones cargadas


## 2. Estado inicial

In [3]:
inicial = []
for nombre, d in regiones.items():
    calendario = pd.date_range(d['Datetime'].min(), d['Datetime'].max(), freq='h')
    inicial.append({
        'region': nombre,
        'filas': len(d),
        'duplicados': int(d.duplicated('Datetime').sum()),
        'horas_faltantes': len(calendario.difference(pd.DatetimeIndex(d['Datetime'].unique()))),
        'ceros_o_negativos': int((d['MW'] <= 0).sum()),
        'nulos': int(d['MW'].isna().sum()),
    })

pd.DataFrame(inicial).set_index('region')

,filas,duplicados,horas_faltantes,ceros_o_negativos,nulos
region,,,,,
AEP,121273,4,27,0,0
COMED,66497,4,11,0,0
DAYTON,121275,4,25,0,0
DEOK,57739,4,9,0,0
DOM,116189,4,23,0,0
DUQ,119068,4,24,0,0
EKPC,45334,4,6,0,0
FE,62874,4,10,1,0
NI,58450,0,14,0,0


## 3. Paso a paso con una región (`FE`)

In [4]:
d0 = regiones['FE']

d1 = unificar_duplicados(d0)
d2 = marcar_valores_sospechosos(d1)
d3 = reindexar_calendario_horario(d2)
d4 = recortar_extremos_vacios(d3)
d5 = interpolar_huecos(d4, limite_horas=LIMITE_HORAS)
validar_limpieza(d5)

Filas eliminadas por duplicados: 4
Valores sospechosos marcados como vacío: 1
Horas agregadas al calendario: 10
Filas recortadas en los extremos: 1
Huecos interpolados: 10 | Vacíos que quedan: 0
Validación correcta: 62879 filas, una por hora, sin vacíos.


,Datetime,MW
0,2011-06-01 02:00:00,8548.0
1,2011-06-01 03:00:00,8121.0
2,2011-06-01 04:00:00,7801.0
3,2011-06-01 05:00:00,7729.0
4,2011-06-01 06:00:00,7968.0
...,...,...
62874,2018-08-02 20:00:00,9866.0
62875,2018-08-02 21:00:00,9656.0
62876,2018-08-02 22:00:00,9532.0
62877,2018-08-02 23:00:00,8872.0


In [5]:
# Filas que fueron vacío antes de interpolar y cómo quedaron
vacias = d4['MW'].isna()
d5[vacias]

,Datetime,MW
3792,2011-11-06 02:00:00,5819.0
6817,2012-03-11 03:00:00,5929.5
12528,2012-11-04 02:00:00,6158.5
15553,2013-03-10 03:00:00,5939.0
21264,2013-11-03 02:00:00,5741.5
24289,2014-03-09 03:00:00,6908.0
33025,2015-03-08 03:00:00,6702.5
41929,2016-03-13 03:00:00,5663.0
50665,2017-03-12 03:00:00,6927.0
59401,2018-03-11 03:00:00,6708.0


## 4. Limpieza de todas las regiones

Los mensajes de cada función se silencian y se registra una bitácora con los conteos. Si una región no pasa la validación, se anota el error y se continúa con las demás.

In [6]:
limpias = {}      # regiones que pasaron la validación
fallidas = {}     # regiones que no la pasaron (se conservan para diagnosticar)
errores = {}
bitacora = []

for nombre, d in regiones.items():
    with redirect_stdout(io.StringIO()):
        d1 = unificar_duplicados(d)
        d2 = marcar_valores_sospechosos(d1)
        d3 = reindexar_calendario_horario(d2)
        d4 = recortar_extremos_vacios(d3)
        d5 = interpolar_huecos(d4, limite_horas=LIMITE_HORAS)

    fila = {
        'region': nombre,
        'filas_originales': len(d),
        'duplicados_unificados': len(d) - len(d1),
        'valores_marcados': int(d2['MW'].isna().sum() - d1['MW'].isna().sum()),
        'horas_agregadas': len(d3) - len(d2),
        'filas_recortadas': len(d3) - len(d4),
        'huecos_interpolados': int(d4['MW'].isna().sum() - d5['MW'].isna().sum()),
        'vacios_restantes': int(d5['MW'].isna().sum()),
        'filas_finales': len(d5),
    }

    try:
        with redirect_stdout(io.StringIO()):
            validar_limpieza(d5)
        limpias[nombre] = d5
        fila['validacion'] = 'OK'
    except ValueError as e:
        fallidas[nombre] = d5
        errores[nombre] = str(e)
        fila['validacion'] = 'ERROR'

    bitacora.append(fila)

bitacora = pd.DataFrame(bitacora).set_index('region')
bitacora

,filas_originales,duplicados_unificados,valores_marcados,horas_agregadas,filas_recortadas,huecos_interpolados,vacios_restantes,filas_finales,validacion
region,,,,,,,,,
AEP,121273,4,0,27,0,27,0,121296,OK
COMED,66497,4,0,11,0,11,0,66504,OK
DAYTON,121275,4,0,25,0,25,0,121296,OK
DEOK,57739,4,0,9,0,9,0,57744,OK
DOM,116189,4,0,23,0,23,0,116208,OK
DUQ,119068,4,0,24,0,24,0,119088,OK
EKPC,45334,4,0,6,0,6,0,45336,OK
FE,62874,4,1,10,1,10,0,62879,OK
NI,58450,0,0,14,0,14,0,58464,OK


In [7]:
# Diagnóstico: qué vacíos quedaron en las regiones que fallaron
if not errores:
    print(f"Las {len(limpias)} regiones pasaron la validación.")
for nombre, mensaje in errores.items():
    print(f"\n{nombre}: {mensaje}")
    display(fallidas[nombre][fallidas[nombre]['MW'].isna()])

Las 12 regiones pasaron la validación.


## 5. Guardado en `data/processed/`

- Un archivo limpio por región: `<REGION>_clean.parquet`.
- Un archivo combinado en formato largo: `consumo_horario.parquet` (`Datetime`, `MW`, `region`).

Solo se guardan las regiones que pasaron la validación. Los datos conservan los MW reales (no se normalizan).

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for nombre, d in limpias.items():
    d.to_parquet(PROCESSED_DIR / f"{nombre}_clean.parquet", index=False)

limpio = (
    pd.concat([d.assign(region=nombre) for nombre, d in limpias.items()], ignore_index=True)
    .sort_values(['region', 'Datetime'])
    .reset_index(drop=True)
)
limpio.to_parquet(PROCESSED_DIR / "consumo_horario.parquet", index=False)

print(f"{len(limpias)} archivos por región + consumo_horario.parquet ({len(limpio):,} filas)")
if errores:
    print("Sin guardar (no pasaron la validación):", list(errores))

12 archivos por región + consumo_horario.parquet (1,090,343 filas)


## 6. Verificación de los archivos guardados

Se leen de nuevo desde disco y se validan. `validar_limpieza` lanza un error si algún archivo no cumple.

In [9]:
archivos = sorted(PROCESSED_DIR.glob("*_clean.parquet"))

resumen = []
for ruta in archivos:
    d = pd.read_parquet(ruta)
    with redirect_stdout(io.StringIO()):
        validar_limpieza(d)
    resumen.append({
        'archivo': ruta.name,
        'filas': len(d),
        'inicio': d['Datetime'].min(),
        'fin': d['Datetime'].max(),
        'vacios': int(d['MW'].isna().sum()),
    })

pd.DataFrame(resumen).set_index('archivo')

,filas,inicio,fin,vacios
archivo,,,,
AEP_clean.parquet,121296,2004-10-01 01:00:00,2018-08-03,0
COMED_clean.parquet,66504,2011-01-01 01:00:00,2018-08-03,0
DAYTON_clean.parquet,121296,2004-10-01 01:00:00,2018-08-03,0
DEOK_clean.parquet,57744,2012-01-01 01:00:00,2018-08-03,0
DOM_clean.parquet,116208,2005-05-01 01:00:00,2018-08-03,0
DUQ_clean.parquet,119088,2005-01-01 01:00:00,2018-08-03,0
EKPC_clean.parquet,45336,2013-06-01 01:00:00,2018-08-03,0
FE_clean.parquet,62879,2011-06-01 02:00:00,2018-08-03,0
NI_clean.parquet,58464,2004-05-01 01:00:00,2011-01-01,0


In [10]:
leido = pd.read_parquet(PROCESSED_DIR / "consumo_horario.parquet")
print(leido.dtypes)
leido.groupby('region').size().rename('filas').to_frame()

Datetime    datetime64[us]
MW                 float64
region                 str
dtype: object


,filas
region,
AEP,121296
COMED,66504
DAYTON,121296
DEOK,57744
DOM,116208
DUQ,119088
EKPC,45336
FE,62879
NI,58464


## Decisiones aplicadas

| Problema | Decisión |
|---|---|
| Horas repetidas (cambio de horario de otoño) | Promediar las lecturas |
| Valores `<= 0` | Marcarlos como vacío y estimarlos |
| Horas ausentes | Reindexar a un calendario horario completo |
| Vacíos en el primer o último registro | Recortarlos (no hay dato para estimarlos) |
| Vacíos cortos | Interpolación lineal hasta `LIMITE_HORAS = 2` |
| Vacíos largos | No se rellenan; la validación los reporta |
| Picos altos de demanda | Se conservan (consumo real) |

**Limitaciones:** la hora de primavera, que no existió en la hora local, se rellena por interpolación; las dos lecturas de otoño se promedian; solo se corrigen valores `<= 0`.

**Siguiente paso:** `03_analisis.ipynb`, leyendo `data/processed/consumo_horario.parquet`.